# Exercise 2b: Change detection in videos

What function is used to convert a color image to a gray-scale image?

In [ ]:
# frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

In [1]:
import time
import cv2
import numpy as np
from skimage.util import img_as_float
from skimage.util import img_as_ubyte

In [2]:
def show_in_moved_window(win_name, img, x, y):
    """
    Show an image in a window, where the position of the window can be given
    """
    cv2.namedWindow(win_name)
    cv2.moveWindow(win_name, x, y)
    cv2.imshow(win_name,img)

In [3]:
def capture_from_camera_and_show_images():
    print("Starting image capture")

    print("Opening connection to camera")
    url = 0
    use_droid_cam = False
    if use_droid_cam:
        url = "http://192.168.1.120:4747/video"
    cap = cv2.VideoCapture(url)
    # cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Cannot open camera")
        exit()

    print("Starting camera loop")
    # Get first image
    ret, frame = cap.read()
    # if frame is read correctly ret is True
    if not ret:
        print("Can't receive frame")
        exit()

    # Transform image to gray scale and then to float, so we can do some processing
    frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    frame_gray = img_as_float(frame_gray)

    # To keep track of frames per second
    start_time = time.time()
    n_frames = 0
    stop = False
    while not stop:
        ret, new_frame = cap.read()
        if not ret:
            print("Can't receive frame. Exiting ...")
            break

        # Transform image to gray scale and then to float, so we can do some processing
        new_frame_gray = cv2.cvtColor(new_frame, cv2.COLOR_BGR2GRAY)
        new_frame_gray = img_as_float(new_frame_gray)

        # Compute difference image
        dif_img = np.abs(new_frame_gray - frame_gray)

        # Binary image by applying a threshold to the difference image
        T = 0.1
        binary_dif_img = dif_img > T

        # Total number of foreground pixels in the binary difference image
        n_foreground_pixels = np.sum(binary_dif_img)

        # Percentage of foreground pixels compared to the total number of pixels in the image
        n_total_pixels = binary_dif_img.size
        F = n_foreground_pixels / n_total_pixels * 100

        # Raise an alarm if the percentage of foreground pixels F is above a alarm threshold A
        A = 0.05
        font = cv2.FONT_HERSHEY_COMPLEX
        if F > A:
            str_out = "Change detected!"
            cv2.putText(new_frame, str_out, (100, 100), font, 1, (0, 0, 255), 1)

        # Keep track of frames-per-second (FPS)
        n_frames = n_frames + 1
        elapsed_time = time.time() - start_time
        fps = int(n_frames / elapsed_time)

        # Put the FPS on the new_frame
        str_out = f"fps: {fps}"
        font = cv2.FONT_HERSHEY_COMPLEX
        cv2.putText(new_frame, str_out, (100, 100), font, 1, 255, 1)

        # Display the resulting frame
        show_in_moved_window('Input', new_frame, 0, 10)
        show_in_moved_window('Input gray', new_frame_gray, 600, 10)
        show_in_moved_window('Difference image', dif_img, 1200, 10)
        show_in_moved_window('Binary difference image', img_as_ubyte(binary_dif_img), 1800, 10)

        # Old frame is updated
        alpha = 0.95
        frame_gray = alpha * frame_gray + (1 - alpha) * new_frame_gray

        if cv2.waitKey(1) == ord('q'):
            stop = True

    print("Stopping image loop")
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
capture_from_camera_and_show_images()

Starting image capture
Opening connection to camera


[ WARN:0@8.963] global cap_gstreamer.cpp:1173 isPipelinePlaying OpenCV | GStreamer warning: GStreamer: pipeline have not been created
2026-02-18 21:07:24.629 python[353:971071] WARNING: AVCaptureDeviceTypeExternal is deprecated for Continuity Cameras. Please use AVCaptureDeviceTypeContinuityCamera and add NSCameraUseContinuityCameraDeviceType to your Info.plist.


Starting camera loop


KeyboardInterrupt: 

: 

#### T (Threshold) = 0.1

- Purpose: Determines which pixels are considered "changed"
- Effect: In binary_dif_img = dif_img > T, any pixel with absolute difference > 0.1 (on 0-1 scale) is marked as changed (white) in the binary image
- Lower T → More sensitive, detects smaller changes
- Higher T → Less sensitive, only detects larger changes

Si el pixel ha canviat o no, comparant el frame anterior i el nou.

#### A (Alarm threshold) = 0.05

- Purpose: Determines when to trigger "Change detected!" alert
- Effect: If more than 5% (A = 0.05) of total pixels have changed, the alarm triggers
- Lower A → Alert triggers more easily (fewer changed pixels needed)
- Higher A → Alert triggers only for significant changes

Si el percentatge de pixels canviats excedeix el threshold escollit o no.

#### alpha = 0.95

- Purpose: Controls how quickly the reference frame adapts (exponential moving average)
- Effect: frame_gray = 0.95 * old_frame + 0.05 * new_frame
- alpha → 1.0: Reference frame barely updates (detects changes vs. original frame)
- alpha → 0.0: Reference frame updates immediately (detects instantaneous motion only)

Quant de la imathe anterior em queda, adaptació gradual. Si alpha=0, cada cop només detecta la motion consecutiva entre dos frames.

